# Bot setup

The Discord bot and the Supabase database are the front end. The bot is a separate distribution, `tri-bot` (`bot/pyproject.toml`), that depends on `tri[render]` and imports only `tri.api`. All network access lives in the bot: Discord, Supabase, dwt.sh, deepwoken.co, pocamind/data and GitHub Pages.

```
pip install ".[render]" ./bot
cp .env.example .env        # fill in; never commit .env
python -m bot
```

Required: `DISCORD_TOKEN`, `SUPABASE_URL`, `SUPABASE_SERVICE_KEY`. Server roles and channels are set in Discord with `/setup`. Every other variable, with its default, is listed in `.env.example` and in the `bot/config.py` docstring; the next cell prints them.

In [ ]:
import bot.config as c
print(c.__doc__)

## Runs

```
python -m bot.gamesource fetch runs/gamedata        # download pocamind/data
python -m tri.cli.fit --builds builds.json --out runs --name live
python -m bot.runstore publish runs/live            # upload to Supabase Storage
python -m bot.runstore fetch runs --name live
```

On start the bot loads the run, reads reviews, locators, authored rules, admin values and categories from the tables and hands them to the library. A new review takes effect at once: the bot builds the next state on a fork (`with_memory`) and swaps it in under its lock while answers keep running on the current state.

## Hosting

`render.yaml` defines one Render background worker (`plan: 1c-2g`) with a 5 GB disk at `/var/data`, `TRI_RUN=/var/data/runs` and `TRI_RUN_REMOTE=latest`. `.python-version` pins Python 3.12 for every Render service, including ones not created from the Blueprint.

## Interactive plots on GitHub Pages

`/plot concept`, `/plot worth` and `/plot atlas` with `interactive:True` post a link instead of an HTML attachment. The page is uploaded to the `gh-pages` branch of `PAGES_REPO` under `plots/<date>/` and served at `https://<owner>.github.io/<repo>/plots/...`; the same plot is uploaded once, and plots older than `PAGES_KEEP_DAYS` are pruned daily. Without `PAGES_REPO` and a token the page is attached as before, and a failed upload falls back to the attachment.

1. Create a fine-grained token for the repository with Contents: read and write, and set `PAGES_TOKEN` (or `GITHUB_TOKEN`).
2. Run `python -m bot.pages check` once: it creates the branch and tries to switch Pages on.
3. If it reports Pages not enabled: Settings → Pages → Deploy from a branch → `gh-pages`, `/ (root)`.

In [ ]:
import subprocess, sys
print(subprocess.run([sys.executable, "-m", "bot.pages", "--help"], capture_output=True, text=True).stdout)